# Minimum Silver Table 1: Syndrome Observations & Shared Tracing
**Course:** TU Delft DSAIT4000 (Data Management & Engineering) — Assignment 1  
**Target Table:** `silver/qec_syndromes/syndrome_observation.parquet`  
**Shared Provenance:** `results/part1/source_trace.parquet`

---

### Objectives & Contracts
1. **Shared Tracing Rule:** Generate stable, deterministic `source_record_id` strings and record complete provenance in `results/part1/source_trace.parquet` using the modular `save_source_traces` helper.
2. **Schema Fidelity:** Build `silver/qec_syndromes/syndrome_observation.parquet` matching all required types (`pa.string()`, `pa.float64()`, `pa.binary()`, `pa.int32()`, `pa.bool_()`, `pa.int64()`).
3. **QEC Invariant Enforcement:**
   - Verify 4 rounds $\times$ 4 checks spatio-temporal syndrome shape.
   - Convert 16 binary bits into a 16-byte binary value (`bytes([bit for round in syn for bit in round])`).
   - Preserve physical multiplicity `quantity` as sample weights (never explode into 70M individual rows).
   - Verify exact reconciliation: **75,598 rows** representing **70,000,000 physical observations**.
4. **Dual-Lake Persistence:** Write Parquet to local disk and upload to MinIO bucket `quantum-lake`.


In [1]:
import ast
import csv
import hashlib
import io
import os
from pathlib import Path
import zipfile

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Platform helpers from starter package
from quantum_lake_student.config import Settings
from quantum_lake_student.connections import minio_client
from quantum_lake_student.tracing import save_source_traces

# Detect if running in container (/workspace) or locally
BASE_DIR = Path("/workspace") if Path("/workspace").exists() else Path(".").resolve()
print(f"Base Directory: {BASE_DIR}")

# Load configuration
settings = Settings.from_environment()
print(f"Lake Backend: {settings.lake_backend}")
print(f"MinIO Endpoint: {settings.s3_endpoint} (Bucket: {settings.s3_bucket})")


Base Directory: /workspace
Lake Backend: minio
MinIO Endpoint: http://minio:9000 (Bucket: quantum-lake)


### Step 1: Environment & Platform Setup Details

1. **Library Imports & Roles:**
   - `ast` & `csv`: Used for resilient parsing. `csv.reader` handles CSV records while `ast.literal_eval` safely deserializes Python tuple representations without executing arbitrary code.
   - `hashlib`: Computes cryptographic SHA-256 checksums of the raw Bronze archive to guarantee auditability and data provenance.
   - `pyarrow` & `pyarrow.parquet`: Enforces strict logical Parquet schemas, column nullability, and compresses output with Zstandard (`zstd`).
2. **Path Auto-Detection (`BASE_DIR`):**
   - Automatically determines whether the notebook is executing inside the Docker workspace container (`/workspace`) or on a local host filesystem (`.`), preventing path discrepancies.
3. **Central Configuration (`Settings`):**
   - Loads runtime environment variables via `Settings.from_environment()`, managing S3 endpoint URLs, bucket names (`quantum-lake`), credentials, and lake backend modes (`minio` vs `local`).
4. **Modular Tracing Helper:**
   - Imports `save_source_traces` from `quantum_lake_student.tracing` to handle lineage registration across the pipeline.


In [2]:
# Locate Bronze zip object (try MinIO first, fallback to local path)
bronze_object_name = "bronze/source=qec_syndromes/syndromes_dataset.zip"
bronze_bytes = None

if settings.lake_backend == "minio":
    try:
        client = minio_client(settings)
        print(f"Fetching '{bronze_object_name}' from MinIO...")
        response = client.get_object(settings.s3_bucket, bronze_object_name)
        bronze_bytes = response.read()
        response.close()
        response.release_conn()
        print(f"Successfully retrieved from MinIO ({len(bronze_bytes):,} bytes)")
    except Exception as e:
        print(f"MinIO fetch warning: {e}. Falling back to local file.")

if bronze_bytes is None:
    # Try common local mounts
    candidates = [
        Path("/course-data/raw/source=qec_syndromes/syndromes_dataset.zip"),
        BASE_DIR.parent / "datasets/student-bundle/core/raw/source=qec_syndromes/syndromes_dataset.zip",
        Path("datasets/student-bundle/core/raw/source=qec_syndromes/syndromes_dataset.zip"),
    ]
    for p in candidates:
        if p.exists():
            print(f"Reading from local path: {p}")
            bronze_bytes = p.read_bytes()
            break

assert bronze_bytes is not None, "Could not locate Bronze syndromes archive!"

# Compute Bronze SHA-256 hash (Required for source_trace.parquet)
bronze_sha256 = hashlib.sha256(bronze_bytes).hexdigest()
print(f"Bronze Archive SHA-256: {bronze_sha256}")
print(f"Bronze Archive Size:   {len(bronze_bytes):,} bytes")


Fetching 'bronze/source=qec_syndromes/syndromes_dataset.zip' from MinIO...
Successfully retrieved from MinIO (358,017 bytes)
Bronze Archive SHA-256: bdfce36a71f04295ac78fb372d9c2e381801c05e3be119f919750ef59026d072
Bronze Archive Size:   358,017 bytes


### Step 2: Bronze Ingestion & Cryptographic Checksum Details

1. **In-Memory Streaming Without Full Extraction:**
   - Adheres to Working Session 1 guidance (*"List stored file paths and archive members without extracting everything"*).
   - Reads the raw archive directly into memory buffer (`bronze_bytes`), avoiding unnecessary temporary disk I/O and leaving Bronze read-only.
2. **Dual-Backend Fallback:**
   - Attempts to stream the object from MinIO object storage (`s3_bucket = "quantum-lake"`).
   - Gracefully falls back to container mount points (`/course-data/raw/...`) if S3 is unavailable.
3. **Audit Hash Computation (`input_sha256`):**
   - Calculates the SHA-256 hash (`bdfce36a71f04295ac78fb372d9c2e381801c05e3be119f919750ef59026d072`).
   - This cryptographic hash binds every transformed row in Silver to the exact byte-level version of the source archive, fulfilling the shared tracing specification.


In [3]:
silver_records = []
trace_records = []

with zipfile.ZipFile(io.BytesIO(bronze_bytes)) as z:
    csv_members = sorted([name for name in z.namelist() if name.endswith(".csv")])
    print(f"Found {len(csv_members)} CSV member files in archive:")

    for member in csv_members:
        stem = Path(member).stem
        pfr_str = member.split("_pfr-")[1].split("_")[0]
        pfr = float(pfr_str)
        exp_id = stem

        with z.open(member) as f:
            reader = csv.reader(io.TextIOWrapper(f))
            header = next(reader)
            assert header in (["labels", "syndromes", "quantity"], ["label", "syndromes", "quantity"]), f"Unexpected header: {header}"

            member_row_count = 0
            member_qty_sum = 0

            for row_idx, row in enumerate(reader):
                assert len(row) == 3, f"Malformed row at {member}:{row_idx}"
                
                # 1. Parse and validate label
                label_val = int(row[0])
                assert label_val in (0, 1), f"Invalid label: {label_val}"
                logical_error_label = bool(label_val)

                # 2. Parse and validate 4x4 syndromes tuple
                syndrome_tuple = ast.literal_eval(row[1])
                assert len(syndrome_tuple) == 4, f"Round count mismatch: {len(syndrome_tuple)}"
                assert all(len(r) == 4 for r in syndrome_tuple), "Check count mismatch"
                
                flat_bits = []
                for round_idx, r in enumerate(syndrome_tuple):
                    for check_idx, bit in enumerate(r):
                        assert bit in (0, 1), f"Non-binary syndrome bit: {bit}"
                        flat_bits.append(bit)
                
                assert len(flat_bits) == 16, "Must have exactly 16 bits"
                syndrome_bytes = bytes(flat_bits)  # exactly 16 one-byte binary values

                # 3. Parse and validate quantity
                quantity = int(row[2])
                assert quantity > 0, f"Non-positive quantity: {quantity}"

                # 4. Generate stable source_record_id
                source_record_id = f"qec_syndromes:{member}:row:{row_idx}"

                # 5. Append to Silver records
                silver_records.append({
                    "source_record_id": source_record_id,
                    "experiment_id": exp_id,
                    "physical_fault_rate": pfr,
                    "syndrome_bits": syndrome_bytes,
                    "round_count": 4,
                    "check_count": 4,
                    "logical_error_label": logical_error_label,
                    "quantity": quantity,
                })

                # 6. Append to Source Trace records
                trace_records.append({
                    "source_record_id": source_record_id,
                    "source_name": "qec_syndromes",
                    "bronze_object": bronze_object_name,
                    "archive_member": member,
                    "record_locator": f"row:{row_idx}",
                    "input_sha256": bronze_sha256,
                })

                member_row_count += 1
                member_qty_sum += quantity

        print(f"  ✓ {member}: {member_row_count:,} rows, {member_qty_sum:,} observations (pfr={pfr:.6f})")

print(f"\nTotal Silver records extracted: {len(silver_records):,}")
print(f"Total Source Trace records:      {len(trace_records):,}")


Found 7 CSV member files in archive:
  ✓ d-3_pfr-0.000010_nb-10M.csv: 68 rows, 10,000,000 observations (pfr=0.000010)
  ✓ d-3_pfr-0.000050_nb-10M.csv: 215 rows, 10,000,000 observations (pfr=0.000050)
  ✓ d-3_pfr-0.000100_nb-10M.csv: 491 rows, 10,000,000 observations (pfr=0.000100)
  ✓ d-3_pfr-0.000500_nb-10M.csv: 1,407 rows, 10,000,000 observations (pfr=0.000500)
  ✓ d-3_pfr-0.001000_nb-10M.csv: 2,854 rows, 10,000,000 observations (pfr=0.001000)
  ✓ d-3_pfr-0.005000_nb-10M.csv: 20,887 rows, 10,000,000 observations (pfr=0.005000)
  ✓ d-3_pfr-0.010000_nb-10M.csv: 49,676 rows, 10,000,000 observations (pfr=0.010000)

Total Silver records extracted: 75,598
Total Source Trace records:      75,598


  ✓ d-3_pfr-0.005000_nb-10M.csv: 20,887 rows, 10,000,000 observations (pfr=0.005000)


  ✓ d-3_pfr-0.010000_nb-10M.csv: 49,676 rows, 10,000,000 observations (pfr=0.010000)

Total Silver records extracted: 75,598
Total Source Trace records:      75,598


### Step 3: Parsing Engine & Invariant Enforcement Details

1. **Iterating Over Archive Members:**
   - Traverses all 7 CSV files in `syndromes_dataset.zip` (`d-3_pfr-0.000010_nb-10M.csv` up to `0.010000`).
   - Extracts the experiment identifier and numerical `physical_fault_rate` ($p$) from each filename.
2. **Robust CSV Reading:**
   - Employs `csv.reader(io.TextIOWrapper(f))` rather than naïve comma splitting (`line.split(',')`), because the nested syndrome tuple string contains internal commas.
3. **Data Quality Assertions Applied to Every Record:**
   - **Label Domain:** Strictly binary (`0` or `1`), stored as boolean `logical_error_label`.
   - **Spatio-Temporal Grid (4x4):** Enforces exactly 4 rounds with 4 stabilizer checks each.
   - **Binary Value Domain:** Verifies every bit $\in \{0, 1\}$.
   - **Binary Packing:** Flattens the 16 bits (round first, check second) and converts them to a 16-byte binary value (`bytes([bit, ...])`) as specified in the Silver contract.
   - **Multiplicity Preservation:** Asserts `quantity > 0` and preserves it as an integer weight. It is **never expanded** into 70M individual rows.
4. **Deterministic Identifier Generation:**
   - Generates `source_record_id = f"qec_syndromes:{member}:row:{row_idx}"`.
   - This ID is deterministic, immutable, and completely independent of database load order or run timestamps.


In [4]:
syndrome_observation_schema = pa.schema([
    ("source_record_id", pa.string()),
    ("experiment_id", pa.string()),
    ("physical_fault_rate", pa.float64()),
    ("syndrome_bits", pa.binary()),
    ("round_count", pa.int32()),
    ("check_count", pa.int32()),
    ("logical_error_label", pa.bool_()),
    ("quantity", pa.int64()),
])

df_silver = pd.DataFrame(silver_records)
table_silver = pa.Table.from_pandas(df_silver, schema=syndrome_observation_schema, preserve_index=False)

print("=== Syndrome Observation Table ===")
print(f"Rows: {table_silver.num_rows:,}, Columns: {table_silver.num_columns}")
print(table_silver.schema)


=== Syndrome Observation Table ===
Rows: 75,598, Columns: 8
source_record_id: string
experiment_id: string
physical_fault_rate: double
syndrome_bits: binary
round_count: int32
check_count: int32
logical_error_label: bool
quantity: int64
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1080


### Step 4: Strict PyArrow Schema & Arrow Table Construction Details

1. **Explicit Schema Enforcement:**
   - Explicitly defines `syndrome_observation_schema` with exact PyArrow types corresponding to `silver-tables.md`:
     - `source_record_id`: `string`
     - `experiment_id`: `string`
     - `physical_fault_rate`: `float64`
     - `syndrome_bits`: `binary`
     - `round_count`: `int32`
     - `check_count`: `int32`
     - `logical_error_label`: `bool`
     - `quantity`: `int64`
2. **Preventing Inferred Type Drift:**
   - Relying on Pandas type inference can cause integers to be cast to floats or strings to objects. `pa.Table.from_pandas(..., schema=schema)` forces exact compliance.
3. **Preserving Clean Storage (`preserve_index=False`):**
   - Strips Pandas' synthetic integer index so it is not stored as an extra metadata column in the final Parquet file.


In [5]:
# Define paths relative to base workspace directory
silver_dir = BASE_DIR / "silver/qec_syndromes"
silver_dir.mkdir(parents=True, exist_ok=True)
silver_parquet_path = silver_dir / "syndrome_observation.parquet"

results_dir = BASE_DIR / "results/part1"
results_dir.mkdir(parents=True, exist_ok=True)
trace_parquet_path = results_dir / "source_trace.parquet"

# Write Silver table with ZSTD compression
pq.write_table(table_silver, silver_parquet_path, compression="zstd")
print(f"✓ Wrote Silver table to: {silver_parquet_path} ({silver_parquet_path.stat().st_size:,} bytes)")

# Upload Silver table to MinIO
if settings.lake_backend == "minio":
    try:
        client = minio_client(settings)
        minio_silver_key = "silver/qec_syndromes/syndrome_observation.parquet"
        client.fput_object(settings.s3_bucket, minio_silver_key, str(silver_parquet_path))
        print(f"✓ Uploaded Silver table to MinIO: {settings.s3_bucket}/{minio_silver_key}")
    except Exception as e:
        print(f"Warning: MinIO upload failed: {e}")

# Save source trace records idempotently using the modular helper
total_traces = save_source_traces(
    new_records=trace_records,
    source_name="qec_syndromes",
    trace_file_path=trace_parquet_path,
    settings=settings,
)
print(f"✓ Saved {total_traces:,} trace records to {trace_parquet_path}")


✓ Wrote Silver table to: /workspace/silver/qec_syndromes/syndrome_observation.parquet (389,379 bytes)
✓ Uploaded Silver table to MinIO: quantum-lake/silver/qec_syndromes/syndrome_observation.parquet
✓ Saved 75,623 trace records to /workspace/results/part1/source_trace.parquet


### Step 5: Atomic Parquet Export & Idempotent Lineage Recording Details

1. **Parquet Compression:**
   - Writes `silver/qec_syndromes/syndrome_observation.parquet` using Zstandard (`compression="zstd"`), achieving high compression ratios (~389 KB for 75,598 rows).
2. **Object Store Synchronization:**
   - Uses `minio_client` to push the written Parquet table to MinIO bucket `quantum-lake` under the canonical path `silver/qec_syndromes/syndrome_observation.parquet`.
3. **Idempotent Lineage Storage via `save_source_traces`:**
   - Calls the modular helper from `quantum_lake_student.tracing`.
   - If `source_trace.parquet` already exists, it refreshes rows for `source_name="qec_syndromes"` while preserving traces from other sources (e.g., `google_qec` and `qasmbench`).
   - Guarantees **idempotency**: repeatedly executing this notebook never produces duplicate rows or inflated file sizes.


In [6]:
# Read back the written parquet table to guarantee write integrity
verified_table = pq.read_table(silver_parquet_path)
df_check = verified_table.to_pandas()

# Invariant Checks
assert len(df_check) == 75598, f"Row count mismatch! Expected 75,598, got {len(df_check)}"
total_obs = df_check["quantity"].sum()
assert total_obs == 70_000_000, f"Observation count mismatch! Expected 70,000,000, got {total_obs}"
assert df_check["source_record_id"].is_unique, "source_record_id must be unique within table"
assert (df_check["round_count"] == 4).all(), "round_count must be 4"
assert (df_check["check_count"] == 4).all(), "check_count must be 4"
assert df_check.isna().sum().sum() == 0, "No null values allowed in Silver"
assert (df_check["syndrome_bits"].str.len() == 16).all(), "Every syndrome_bits entry must be 16 bytes"

print("✓ All 6 Data Quality Invariants Passed!")
print(f"  • Rows:         {len(df_check):,}")
print(f"  • Observations: {total_obs:,}")
print(f"  • Nulls:        {df_check.isna().sum().sum()}")


✓ All 6 Data Quality Invariants Passed!
  • Rows:         75,598
  • Observations: 70,000,000
  • Nulls:        0


### Step 6: Post-Build Verification & Integrity Guarantees Details

1. **Disk-Read Validation:**
   - Re-reads the physical Parquet file from disk into PyArrow to confirm file readability and serialization integrity.
2. **Reconciliation Invariants Tested:**
   - **Row Count:** Exactly **75,598 rows** verified.
   - **Weighted Observations:** Exactly **70,000,000 observations** verified via `quantity.sum()`.
   - **Primary Key Uniqueness:** Verifies zero duplicate `source_record_id` values.
   - **Structural Invariants:** Verifies `round_count == 4` and `check_count == 4` across 100% of rows.
   - **Nullability Rules:** Asserts 0 nulls across the entire table.
   - **Binary Length:** Asserts all `syndrome_bits` values are exactly 16 bytes long.
